In [1]:
%cd ..

/home/oleg/audio-llm-yandex-camp


In [14]:
from src.asr_eval.align.data import MatchesList
from src.asr_eval.align.parsing import split_text_into_tokens
from src.asr_eval.align.recursive import align
from src.asr_eval.utils.formatting import Formatting, FormattingSpan, apply_ansi_formatting
from termcolor import colored

truth = 'чипи чипи чапа чапа'
preds = {
    'model1': 'чип чапа',
    'model2': 'чипи чипи чапа1 чапа',
    'model3': 'чипи-чипи-чапа-чапа-чапа',
}

true_tokens = split_text_into_tokens(truth)

def display_alignment(alignment: MatchesList):
    true_line: str = 'TRUE: '
    pred_line: str = 'PRED: '
    color_spans: list[FormattingSpan] = []

    for m in alignment.matches:
        true_word = str(m.true.value) if m.true is not None else ''
        pred_word = str(m.pred.value) if m.pred is not None else ''
        
        display_length = max(len(true_word), len(pred_word))
        true_word = true_word.ljust(display_length)
        pred_word = pred_word.ljust(display_length)
        
        pred_span = (len(pred_line), len(pred_line) + len(pred_word))
        
        true_line += true_word + ' '
        pred_line += pred_word + ' '
        
        match m.status:
            case 'correct':
                pass
            case 'replacement' | 'deletion' | 'insertion':
                color_spans.append(FormattingSpan(Formatting(on_color='on_yellow'), *pred_span))
        
    print(colored(true_line, attrs=['bold']))
    print(apply_ansi_formatting(pred_line, color_spans))

for model_name, pred in preds.items():
    pred_tokens = split_text_into_tokens(pred)

    alignment = align(true_tokens, pred_tokens)

    display_alignment(alignment)
    print()

TRUE: чипи чипи чапа чапа 
PRED: чип       чапа      

TRUE: чипи чипи чапа  чапа 
PRED: чипи чипи чапа1 чапа 

TRUE: чипи чипи чапа чапа      
PRED: чипи чипи чапа чапа чапа 

